# Notebook 2: Scam-Archetype Classifier

This notebook implements Phase 2a from the proposal:

1. A TF-IDF + logistic-regression baseline
2. A DistilBERT preliminary experiment
3. An optional RoBERTa comparison
4. Class-imbalance handling with weighted cross-entropy
5. Accuracy, macro F1, per-class metrics, confusion matrix, and confidence analysis

Do not report final model scores while `labels_are_provisional` is true in the discovery output.


## 1. Install dependencies


In [ ]:
%pip install -q "transformers>=4.46,<6" "datasets>=3,<5" "accelerate>=1,<2" \
  "torch>=2.2" "scikit-learn>=1.5,<2" "pandas>=2.2,<3" "matplotlib>=3.8,<4"


## 2. Imports and experiment configuration


In [ ]:
import inspect, json, random, sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.metrics import classification_metrics, expected_calibration_error

with open(PROJECT_ROOT / "configs/project_config.json", encoding="utf-8") as f:
    CONFIG = json.load(f)

SEED = CONFIG["seed"]
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DATA_DIR = PROJECT_ROOT / CONFIG["processed_dir"]
RESULTS_DIR = PROJECT_ROOT / "results" / "classifier"
MODEL_DIR = PROJECT_ROOT / "models" / "classifier"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TEXT_COLUMN = "clean_text"
LABEL_COLUMN = "archetype_label"
MAX_LENGTH = 384

# Preliminary run: train both models on a small stratified sample for one epoch.
FAST_DEV_RUN = True
RUN_ALL_MODELS = True
USE_CLASS_WEIGHTS = True
MODEL_CANDIDATES = CONFIG["classifier_models"]


## 3. Load the fixed splits


In [ ]:
train_df = pd.read_csv(DATA_DIR / "split_train.csv.gz")
val_df = pd.read_csv(DATA_DIR / "split_val.csv.gz")
test_df = pd.read_csv(DATA_DIR / "split_test.csv.gz")

for name, frame in {"train": train_df, "val": val_df, "test": test_df}.items():
    missing = {TEXT_COLUMN, LABEL_COLUMN}.difference(frame.columns)
    if missing:
        raise ValueError(f"{name} split is missing columns: {sorted(missing)}")

if "labels_are_provisional" in train_df and train_df["labels_are_provisional"].astype(bool).any():
    print("WARNING: discovery labels are provisional. Use this run only to verify the code path.")

labels = sorted(train_df[LABEL_COLUMN].unique())
label2id = {label: index for index, label in enumerate(labels)}
id2label = {index: label for label, index in label2id.items()}

for frame in (train_df, val_df, test_df):
    frame["label"] = frame[LABEL_COLUMN].map(label2id)

print("Classes:", len(labels))
print(train_df[LABEL_COLUMN].value_counts())


## 4. Benchmark 1: TF-IDF + logistic regression

This inexpensive baseline is important. A Transformer should beat it on macro F1 or provide a clear
reason to accept the extra training and inference cost.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from sklearn.pipeline import Pipeline

baseline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.95,
        max_features=100_000,
        sublinear_tf=True,
    )),
    ("classifier", LogisticRegression(
        max_iter=2_000,
        class_weight="balanced",
        n_jobs=-1,
        random_state=SEED,
    )),
])

baseline.fit(train_df[TEXT_COLUMN], train_df["label"])
baseline_pred = baseline.predict(test_df[TEXT_COLUMN])
baseline_metrics = classification_metrics(test_df["label"], baseline_pred)
print(baseline_metrics)
print(classification_report(test_df["label"], baseline_pred, target_names=labels, zero_division=0))

fig, ax = plt.subplots(figsize=(10, 9))
ConfusionMatrixDisplay.from_predictions(
    test_df["label"], baseline_pred, display_labels=labels, xticks_rotation=90,
    normalize="true", values_format=".2f", ax=ax
)
ax.set_title("TF-IDF baseline: row-normalized confusion matrix")
plt.tight_layout()
plt.show()


## 5. Prepare Hugging Face datasets


In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, DataCollatorWithPadding


def stratified_cap(frame, cap_per_class):
    return (
        frame.groupby(LABEL_COLUMN, group_keys=False)
        .apply(lambda group: group.sample(min(len(group), cap_per_class), random_state=SEED))
        .sample(frac=1.0, random_state=SEED)
        .reset_index(drop=True)
    )

if FAST_DEV_RUN:
    train_run = stratified_cap(train_df, 400)
    val_run = stratified_cap(val_df, 100)
    test_run = stratified_cap(test_df, 100)
    EPOCHS = 1
else:
    train_run, val_run, test_run = train_df, val_df, test_df
    EPOCHS = 3

print("Run sizes:", len(train_run), len(val_run), len(test_run))


## 6. Weighted Trainer and reusable experiment function


In [ ]:
from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoModelForSequenceClassification,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels_tensor = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = self.class_weights.to(logits.device) if self.class_weights is not None else None
        loss = torch.nn.CrossEntropyLoss(weight=weights)(logits, labels_tensor)
        return (loss, outputs) if return_outputs else loss


def make_training_args(output_dir):
    kwargs = dict(
        output_dir=str(output_dir),
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=2,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        warmup_ratio=0.10,
        logging_strategy="steps",
        logging_steps=25,
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        save_total_limit=2,
        report_to="none",
        fp16=torch.cuda.is_available(),
        seed=SEED,
        data_seed=SEED,
    )
    signature = inspect.signature(TrainingArguments.__init__).parameters
    kwargs["eval_strategy" if "eval_strategy" in signature else "evaluation_strategy"] = "epoch"
    return TrainingArguments(**kwargs)


def compute_metrics(eval_prediction):
    logits, true_labels = eval_prediction
    pred_labels = np.argmax(logits, axis=-1)
    return classification_metrics(true_labels, pred_labels)


def run_transformer_experiment(model_name):
    slug = model_name.split("/")[-1]
    output_dir = MODEL_DIR / slug
    tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)

    def tokenize(batch):
        return tokenizer(batch[TEXT_COLUMN], truncation=True, max_length=MAX_LENGTH)

    def to_dataset(frame):
        ds = Dataset.from_pandas(frame[[TEXT_COLUMN, "label"]], preserve_index=False)
        ds = ds.map(tokenize, batched=True, remove_columns=[TEXT_COLUMN])
        return ds.rename_column("label", "labels")

    train_ds = to_dataset(train_run)
    val_ds = to_dataset(val_run)
    test_ds = to_dataset(test_run)

    model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=len(labels),
        id2label=id2label,
        label2id=label2id,
    )
    collator = DataCollatorWithPadding(tokenizer=tokenizer)

    class_weights = None
    if USE_CLASS_WEIGHTS:
        weights = compute_class_weight(
            class_weight="balanced",
            classes=np.arange(len(labels)),
            y=train_run["label"].to_numpy(),
        )
        class_weights = torch.tensor(weights, dtype=torch.float32)

    trainer_kwargs = dict(
        model=model,
        args=make_training_args(output_dir),
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        class_weights=class_weights,
    )
    trainer_signature = inspect.signature(Trainer.__init__).parameters
    if "processing_class" in trainer_signature:
        trainer_kwargs["processing_class"] = tokenizer
    else:
        trainer_kwargs["tokenizer"] = tokenizer

    trainer = WeightedTrainer(**trainer_kwargs)
    trainer.train()
    prediction = trainer.predict(test_ds)
    logits = prediction.predictions
    probabilities = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    pred_labels = probabilities.argmax(axis=1)
    metrics = classification_metrics(prediction.label_ids, pred_labels)
    metrics["ece"] = expected_calibration_error(prediction.label_ids, probabilities)
    metrics["model"] = model_name
    metrics["fast_dev_run"] = FAST_DEV_RUN

    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    return trainer, prediction.label_ids, pred_labels, probabilities, metrics


## 7. Run DistilBERT and, optionally, RoBERTa


In [ ]:
models_to_run = MODEL_CANDIDATES if RUN_ALL_MODELS else MODEL_CANDIDATES[:1]
transformer_runs = {}
results = [{"model": "tfidf_logistic_regression", **baseline_metrics, "ece": np.nan, "fast_dev_run": False}]

for model_name in models_to_run:
    print("\n" + "=" * 80)
    print("Training", model_name)
    trainer, y_true, y_pred, probabilities, metrics = run_transformer_experiment(model_name)
    transformer_runs[model_name] = (trainer, y_true, y_pred, probabilities)
    results.append(metrics)

benchmark_results = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
benchmark_results.to_csv(RESULTS_DIR / "model_benchmark.csv", index=False)
benchmark_results


## 8. Detailed evaluation and confidence analysis


In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

best_transformer_name = benchmark_results.loc[
    benchmark_results["model"] != "tfidf_logistic_regression", "model"
].iloc[0]
_, y_true, y_pred, probabilities = transformer_runs[best_transformer_name]

report = classification_report(
    y_true, y_pred, target_names=labels, output_dict=True, zero_division=0
)
pd.DataFrame(report).T.to_csv(RESULTS_DIR / "classification_report.csv")
display(pd.DataFrame(report).T)

fig, ax = plt.subplots(figsize=(10, 9))
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=labels, xticks_rotation=90,
    normalize="true", values_format=".2f", ax=ax
)
ax.set_title(f"{best_transformer_name}: row-normalized confusion matrix")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()

confidence = probabilities.max(axis=1)
confidence_table = pd.DataFrame({
    "confidence": confidence,
    "correct": (y_true == y_pred).astype(int),
})
confidence_table["bin"] = pd.cut(confidence_table["confidence"], bins=np.linspace(0, 1, 11), include_lowest=True)
calibration = confidence_table.groupby("bin", observed=False).agg(
    count=("correct", "size"),
    accuracy=("correct", "mean"),
    mean_confidence=("confidence", "mean"),
)
calibration.to_csv(RESULTS_DIR / "confidence_calibration.csv")
calibration


## 9. Final-run checklist

For the final reported experiment:

- Complete human topic labels in Notebook 1 and rerun the splits
- Set `FAST_DEV_RUN = False`
- Train both DistilBERT and RoBERTa using the same splits
- Compare weighted-loss training against an unweighted run
- Report accuracy, macro F1, per-archetype precision/recall/F1, confusion matrix, and ECE
- Save the best checkpoint and its label mapping


In [ ]:
with open(RESULTS_DIR / "label_mapping.json", "w", encoding="utf-8") as f:
    json.dump({"label2id": label2id, "id2label": id2label}, f, indent=2)
print("Saved classifier outputs to", RESULTS_DIR)
